# Problem 18: The Smart Container Terminal Integration Problem

## Tier 2: Heuristic Implementation

### Problem Introduction

In Tier 1, we formulated the Smart Container Terminal Integration Problem as a Mixed-Integer Programming (MIP) model. While MIP provides optimal solutions, it can be computationally expensive for large-scale terminal operations. **Tier 2 introduces heuristic approaches** that provide fast, high-quality solutions suitable for real-time decision making.

### Why Heuristics vs MIP?

**Advantages of Heuristics:**
1. **Speed**: Heuristics find solutions in seconds vs minutes/hours for MIP
2. **Scalability**: Can handle much larger terminal instances
3. **Robustness**: Less sensitive to problem size and complexity
4. **Implementation**: Easier to implement and maintain

**Disadvantages:**
1. **Optimality Gap**: May not find the true optimal solution
2. **Parameter Tuning**: Performance depends on parameter settings
3. **Problem Specific**: Heuristics are often problem-specific

### Heuristic Approaches Implemented:
1. **Priority-Based Dispatching**: Assign tasks based on priority rules
2. **Greedy Assignment**: Assign each task to the best available equipment
3. **Look-Ahead Scheduling**: Consider future task implications
4. **Load Balancing**: Distribute workload evenly across equipment

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import heapq
import time
from collections import defaultdict

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All packages imported successfully!")

### Data Structures (Same as Tier 1)

In [ ]:
@dataclass
class Container:
    """Represents a container to be processed"""
    id: int
    size: str  # '20ft' or '40ft'
    weight: float  # tons
    origin: str  # 'vessel' or 'yard'
    destination: str  # 'vessel' or 'yard'
    priority: int  # 1=high, 2=medium, 3=low
    earliest_time: int  # earliest time processing can start
    latest_time: int  # latest time processing must complete

@dataclass
class Equipment:
    """Represents terminal equipment"""
    id: int
    type: str  # 'AGV', 'QC', or 'YC'
    capacity: float  # lifting capacity in tons
    speed: float  # travel speed (m/min for AGV, lifts/min for cranes)
    position: Tuple[float, float]  # (x, y) coordinates
    available_times: List[int]  # time periods when equipment is available

@dataclass
class Task:
    """Represents a processing task"""
    id: int
    container_id: int
    equipment_type: str  # required equipment type
    processing_time: int  # time required in minutes
    location: Tuple[float, float]  # task location
    precedence: List[int]  # tasks that must be completed first

print("✅ Data structures defined!")

### Instance Generation (Same as Tier 1)

In [ ]:
def generate_terminal_instance():
    """Generate a realistic terminal integration instance"""
    
    # Generate containers
    containers = []
    container_id = 1
    
    # Import containers (from yard to vessel)
    for i in range(10):  # 10 import containers
        containers.append(Container(
            id=container_id,
            size=np.random.choice(['20ft', '40ft'], p=[0.6, 0.4]),
            weight=np.random.uniform(10, 30),
            origin='yard',
            destination='vessel',
            priority=np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2]),
            earliest_time=np.random.randint(0, 30),
            latest_time=np.random.randint(60, 120)
        ))
        container_id += 1
    
    # Export containers (from vessel to yard)
    for i in range(8):  # 8 export containers
        containers.append(Container(
            id=container_id,
            size=np.random.choice(['20ft', '40ft'], p=[0.6, 0.4]),
            weight=np.random.uniform(10, 30),
            origin='vessel',
            destination='yard',
            priority=np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2]),
            earliest_time=np.random.randint(0, 30),
            latest_time=np.random.randint(60, 120)
        ))
        container_id += 1
    
    # Generate equipment
    equipment = []
    
    # AGVs (Automated Guided Vehicles)
    for i in range(5):  # 5 AGVs
        equipment.append(Equipment(
            id=i+1,
            type='AGV',
            capacity=40,  # 40 tons
            speed=200,  # 200 m/min
            position=(np.random.uniform(0, 500), np.random.uniform(0, 300)),
            available_times=list(range(0, 1440))  # 24 hours in minutes
        ))
    
    # Quay Cranes (vessel operations)
    for i in range(3):  # 3 QCs
        equipment.append(Equipment(
            id=6+i,
            type='QC',
            capacity=50,  # 50 tons
            speed=2,  # 2 lifts/min
            position=(200 + i*100, 50),  # Along quay
            available_times=list(range(0, 1440))
        ))
    
    # Yard Cranes (yard operations)
    for i in range(4):  # 4 YCs
        equipment.append(Equipment(
            id=9+i,
            type='YC',
            capacity=40,  # 40 tons
            speed=1.5,  # 1.5 lifts/min
            position=(100 + i*100, 200),  # In yard blocks
            available_times=list(range(0, 1440))
        ))
    
    # Generate tasks
    tasks = []
    task_id = 1
    
    for container in containers:
        # Each container requires 3 tasks: pickup, transport, delivery
        
        # Task 1: Pickup (YC for import, QC for export)
        pickup_type = 'YC' if container.origin == 'yard' else 'QC'
        pickup_loc = (np.random.uniform(50, 450), 200) if pickup_type == 'YC' else (250, 50)
        
        tasks.append(Task(
            id=task_id,
            container_id=container.id,
            equipment_type=pickup_type,
            processing_time=np.random.randint(3, 8),
            location=pickup_loc,
            precedence=[]
        ))
        task_id += 1
        
        # Task 2: Transport (AGV)
        tasks.append(Task(
            id=task_id,
            container_id=container.id,
            equipment_type='AGV',
            processing_time=np.random.randint(5, 15),
            location=(np.random.uniform(100, 400), np.random.uniform(100, 200)),
            precedence=[task_id-1]  # Must complete pickup first
        ))
        task_id += 1
        
        # Task 3: Delivery (QC for import, YC for export)
        delivery_type = 'QC' if container.destination == 'vessel' else 'YC'
        delivery_loc = (np.random.uniform(50, 450), 200) if delivery_type == 'YC' else (300, 50)
        
        tasks.append(Task(
            id=task_id,
            container_id=container.id,
            equipment_type=delivery_type,
            processing_time=np.random.randint(3, 8),
            location=delivery_loc,
            precedence=[task_id-1]  # Must complete transport first
        ))
        task_id += 1
    
    return containers, equipment, tasks

# Generate the instance
containers, equipment, tasks = generate_terminal_instance()

print(f"📦 Generated {len(containers)} containers")
print(f"🤖 Generated {len(equipment)} equipment units")
print(f"📋 Generated {len(tasks)} tasks")
print(f"⏰ Planning horizon: 24 hours (1440 minutes)")

### Heuristic 1: Priority-Based Dispatching

This heuristic assigns tasks based on priority rules considering container priority, deadlines, and equipment availability.

In [ ]:
def priority_based_dispatching(containers, equipment, tasks):
    """Priority-based dispatching heuristic"""
    
    print("🚀 Priority-Based Dispatching Heuristic")
    start_time = time.time()
    
    # Initialize equipment schedules
    equipment_schedule = {eq.id: [] for eq in equipment}
    equipment_available_time = {eq.id: 0 for eq in equipment}
    
    # Calculate task priorities
    task_priorities = {}
    for task in tasks:
        container = next(c for c in containers if c.id == task.container_id)
        
        # Priority score: lower is better (higher priority)
        # Consider container priority, deadline urgency, and processing time
        urgency = (container.latest_time - container.earliest_time) / container.latest_time
        priority_score = container.priority * 10 + urgency * 5 + task.processing_time
        
        task_priorities[task.id] = priority_score
    
    # Sort tasks by priority (lower score = higher priority)
    sorted_tasks = sorted(tasks, key=lambda t: task_priorities[t.id])
    
    solution = []
    completed_tasks = set()
    
    # Assign tasks in priority order
    for task in sorted_tasks:
        # Check if all predecessors are completed
        if all(pred_id in completed_tasks for pred_id in task.precedence):
            
            # Find available equipment of correct type
            suitable_equipment = [eq for eq in equipment if eq.type == task.equipment_type]
            
            # Select equipment with earliest availability
            best_equipment = min(suitable_equipment, 
                                key=lambda eq: equipment_available_time[eq.id])
            
            # Calculate start time
            container = next(c for c in containers if c.id == task.container_id)
            earliest_start = max(equipment_available_time[best_equipment.id], 
                                container.earliest_time)
            
            # Check capacity constraint
            if container.weight <= best_equipment.capacity:
                
                # Schedule the task
                start_time_task = earliest_start
                end_time_task = start_time_task + task.processing_time
                
                # Check if within deadline
                if end_time_task <= container.latest_time:
                    
                    solution.append({
                        'task_id': task.id,
                        'container_id': container.id,
                        'equipment_id': best_equipment.id,
                        'equipment_type': best_equipment.type,
                        'start_time': start_time_task,
                        'end_time': end_time_task,
                        'processing_time': task.processing_time,
                        'delay': 0,
                        'location': task.location,
                        'priority_score': task_priorities[task.id]
                    })
                    
                    # Update equipment availability
                    equipment_available_time[best_equipment.id] = end_time_task
                    completed_tasks.add(task.id)
                else:
                    # Task will be delayed
                    delay = end_time_task - container.latest_time
                    
                    solution.append({
                        'task_id': task.id,
                        'container_id': container.id,
                        'equipment_id': best_equipment.id,
                        'equipment_type': best_equipment.type,
                        'start_time': start_time_task,
                        'end_time': end_time_task,
                        'processing_time': task.processing_time,
                        'delay': delay,
                        'location': task.location,
                        'priority_score': task_priorities[task.id]
                    })
                    
                    equipment_available_time[best_equipment.id] = end_time_task
                    completed_tasks.add(task.id)
    
    end_time = time.time()
    solve_time = end_time - start_time
    
    # Calculate total cost
    total_cost = len(solution) * 1.0  # Equipment utilization cost
    total_cost += sum(task['delay'] * 10 for task in solution)  # Delay cost
    
    print(f"⏱️ Solve time: {solve_time:.4f} seconds")
    print(f"💰 Total cost: ${total_cost:.2f}")
    print(f"📋 Scheduled tasks: {len(solution)}")
    
    return solution, total_cost, solve_time

print("✅ Priority-based dispatching heuristic defined!")

### Heuristic 2: Greedy Assignment with Load Balancing

This heuristic assigns each task to the equipment that minimizes a cost function while balancing workload.

In [ ]:
def greedy_assignment_balancing(containers, equipment, tasks):
    """Greedy assignment with load balancing heuristic"""
    
    print("🎯 Greedy Assignment with Load Balancing")
    start_time = time.time()
    
    # Initialize equipment schedules and load tracking
    equipment_schedule = {eq.id: [] for eq in equipment}
    equipment_load = {eq.id: 0 for eq in equipment}  # Total processing time assigned
    equipment_available_time = {eq.id: 0 for eq in equipment}
    
    solution = []
    completed_tasks = set()
    
    # Sort tasks by earliest start time (FCFS with some flexibility)
    sorted_tasks = sorted(tasks, key=lambda t: min(c.earliest_time for c in containers if c.id == t.container_id))
    
    for task in sorted_tasks:
        # Check if all predecessors are completed
        if not all(pred_id in completed_tasks for pred_id in task.precedence):
            continue
        
        container = next(c for c in containers if c.id == task.container_id)
        suitable_equipment = [eq for eq in equipment if eq.type == task.equipment_type and container.weight <= eq.capacity]
        
        if not suitable_equipment:
            continue
        
        # Calculate cost for each suitable equipment
        equipment_costs = []
        for eq in suitable_equipment:
            # Cost components:
            # 1. Waiting time (equipment availability)
            wait_time = max(0, equipment_available_time[eq.id] - container.earliest_time)
            
            # 2. Load balancing penalty (prefer less loaded equipment)
            avg_load = sum(equipment_load.values()) / len(equipment_load)
            load_penalty = (equipment_load[eq.id] - avg_load) / avg_load if avg_load > 0 else 0
            
            # 3. Distance penalty (travel time approximation)
            distance = abs(eq.position[0] - task.location[0]) + abs(eq.position[1] - task.location[1])
            distance_penalty = distance / 1000  # Normalize
            
            # Total cost (lower is better)
            total_cost = wait_time + load_penalty * 10 + distance_penalty * 5
            
            equipment_costs.append((eq, total_cost))
        
        # Select equipment with minimum cost
        best_equipment, min_cost = min(equipment_costs, key=lambda x: x[1])
        
        # Calculate start time
        earliest_start = max(equipment_available_time[best_equipment.id], container.earliest_time)
        
        # Schedule the task
        start_time_task = earliest_start
        end_time_task = start_time_task + task.processing_time
        delay = max(0, end_time_task - container.latest_time)
        
        solution.append({
            'task_id': task.id,
            'container_id': container.id,
            'equipment_id': best_equipment.id,
            'equipment_type': best_equipment.type,
            'start_time': start_time_task,
            'end_time': end_time_task,
            'processing_time': task.processing_time,
            'delay': delay,
            'location': task.location,
            'assignment_cost': min_cost
        })
        
        # Update equipment state
        equipment_available_time[best_equipment.id] = end_time_task
        equipment_load[best_equipment.id] += task.processing_time
        completed_tasks.add(task.id)
    
    end_time = time.time()
    solve_time = end_time - start_time
    
    # Calculate total cost
    total_cost = len(solution) * 1.0  # Equipment utilization cost
    total_cost += sum(task['delay'] * 10 for task in solution)  # Delay cost
    
    print(f"⏱️ Solve time: {solve_time:.4f} seconds")
    print(f"💰 Total cost: ${total_cost:.2f}")
    print(f"📋 Scheduled tasks: {len(solution)}")
    
    return solution, total_cost, solve_time

print("✅ Greedy assignment with load balancing heuristic defined!")

### Heuristic 3: Look-Ahead Scheduling

This heuristic considers the impact of current decisions on future tasks to make more informed choices.

In [ ]:
def look_ahead_scheduling(containers, equipment, tasks, look_ahead_window=5):
    """Look-ahead scheduling heuristic"""
    
    print("🔮 Look-Ahead Scheduling Heuristic")
    start_time = time.time()
    
    # Initialize equipment schedules
    equipment_schedule = {eq.id: [] for eq in equipment}
    equipment_available_time = {eq.id: 0 for eq in equipment}
    
    solution = []
    completed_tasks = set()
    remaining_tasks = tasks.copy()
    
    while remaining_tasks:
        # Find tasks ready to be scheduled (predecessors completed)
        ready_tasks = [t for t in remaining_tasks 
                      if all(pred_id in completed_tasks for pred_id in t.precedence)]
        
        if not ready_tasks:
            break  # No more tasks can be scheduled
        
        best_task = None
        best_equipment = None
        best_score = float('inf')
        
        # Evaluate each ready task with look-ahead
        for task in ready_tasks[:look_ahead_window]:  # Limit look-ahead for performance
            container = next(c for c in containers if c.id == task.container_id)
            suitable_equipment = [eq for eq in equipment 
                                 if eq.type == task.equipment_type and container.weight <= eq.capacity]
            
            for eq in suitable_equipment:
                # Calculate immediate impact
                earliest_start = max(equipment_available_time[eq.id], container.earliest_time)
                end_time = earliest_start + task.processing_time
                immediate_delay = max(0, end_time - container.latest_time)
                
                # Look-ahead impact: check effect on dependent tasks
                dependent_tasks = [t for t in remaining_tasks if task.id in t.precedence]
                future_impact = 0
                
                for dep_task in dependent_tasks[:2]:  # Limit look-ahead depth
                    dep_container = next(c for c in containers if c.id == dep_task.container_id)
                    dep_eq_available = end_time  # This equipment will be available at end_time
                    
                    # Find best equipment for dependent task
                    dep_suitable_eq = [e for e in equipment 
                                     if e.type == dep_task.equipment_type and dep_container.weight <= e.capacity]
                    
                    if dep_suitable_eq:
                        best_dep_eq = min(dep_suitable_eq, 
                                        key=lambda e: max(equipment_available_time[e.id], dep_eq_available))
                        
                        dep_start = max(equipment_available_time[best_dep_eq.id], dep_eq_available, 
                                      dep_container.earliest_time)
                        dep_end = dep_start + dep_task.processing_time
                        dep_delay = max(0, dep_end - dep_container.latest_time)
                        
                        future_impact += dep_delay
                
                # Total score (lower is better)
                total_score = immediate_delay + future_impact * 0.5  # Weight future impact less
                
                if total_score < best_score:
                    best_score = total_score
                    best_task = task
                    best_equipment = eq
        
        if best_task and best_equipment:
            # Schedule the best task
            container = next(c for c in containers if c.id == best_task.container_id)
            earliest_start = max(equipment_available_time[best_equipment.id], container.earliest_time)
            end_time = earliest_start + best_task.processing_time
            delay = max(0, end_time - container.latest_time)
            
            solution.append({
                'task_id': best_task.id,
                'container_id': container.id,
                'equipment_id': best_equipment.id,
                'equipment_type': best_equipment.type,
                'start_time': earliest_start,
                'end_time': end_time,
                'processing_time': best_task.processing_time,
                'delay': delay,
                'location': best_task.location,
                'look_ahead_score': best_score
            })
            
            # Update state
            equipment_available_time[best_equipment.id] = end_time
            completed_tasks.add(best_task.id)
            remaining_tasks.remove(best_task)
        else:
            break  # No valid assignment found
    
    end_time = time.time()
    solve_time = end_time - start_time
    
    # Calculate total cost
    total_cost = len(solution) * 1.0  # Equipment utilization cost
    total_cost += sum(task['delay'] * 10 for task in solution)  # Delay cost
    
    print(f"⏱️ Solve time: {solve_time:.4f} seconds")
    print(f"💰 Total cost: ${total_cost:.2f}")
    print(f"📋 Scheduled tasks: {len(solution)}")
    
    return solution, total_cost, solve_time

print("✅ Look-ahead scheduling heuristic defined!")

### Run All Heuristics and Compare

In [ ]:
# Run all heuristics
print("🚀 RUNNING ALL HEURISTICS")
print("=" * 60)

# Heuristic 1: Priority-Based Dispatching
print("\n1️⃣ PRIORITY-BASED DISPATCHING")
print("-" * 40)
solution1, cost1, time1 = priority_based_dispatching(containers, equipment, tasks)

# Heuristic 2: Greedy Assignment with Load Balancing
print("\n2️⃣ GREEDY ASSIGNMENT WITH LOAD BALANCING")
print("-" * 40)
solution2, cost2, time2 = greedy_assignment_balancing(containers, equipment, tasks)

# Heuristic 3: Look-Ahead Scheduling
print("\n3️⃣ LOOK-AHEAD SCHEDULING")
print("-" * 40)
solution3, cost3, time3 = look_ahead_scheduling(containers, equipment, tasks)

# Summary comparison
print("\n📊 HEURISTIC COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Heuristic':<35} {'Cost':<10} {'Time (s)':<10} {'Tasks':<10}")
print("-" * 60)
print(f"{'Priority-Based Dispatching':<35} ${cost1:<9.2f} {time1:<9.4f} {len(solution1):<10}")
print(f"{'Greedy Assignment + Balancing':<35} ${cost2:<9.2f} {time2:<9.4f} {len(solution2):<10}")
print(f"{'Look-Ahead Scheduling':<35} ${cost3:<9.2f} {time3:<9.4f} {len(solution3):<10}")

# Find best heuristic
heuristic_results = [
    ('Priority-Based', solution1, cost1, time1),
    ('Greedy + Balancing', solution2, cost2, time2),
    ('Look-Ahead', solution3, cost3, time3)
]

best_heuristic = min(heuristic_results, key=lambda x: x[2])
print(f"\n🏆 BEST HEURISTIC: {best_heuristic[0]} (Cost: ${best_heuristic[2]:.2f})")

### Detailed Analysis of Best Heuristic

In [ ]:
def analyze_heuristic_solution(solution, heuristic_name):
    """Detailed analysis of heuristic solution"""
    
    if not solution:
        print(f"❌ No solution found for {heuristic_name}!")
        return
    
    df_solution = pd.DataFrame(solution)
    
    print(f"\n📈 DETAILED ANALYSIS: {heuristic_name}")
    print("=" * 50)
    
    # Equipment utilization
    print(f"\n🤖 EQUIPMENT UTILIZATION:")
    for eq_type in ['AGV', 'QC', 'YC']:
        eq_tasks = df_solution[df_solution['equipment_type'] == eq_type]
        if not eq_tasks.empty:
            total_processing = eq_tasks['processing_time'].sum()
            eq_count = len([e for e in equipment if e.type == eq_type])
            horizon_minutes = 1440
            utilization = (total_processing / (eq_count * horizon_minutes)) * 100
            print(f"  {eq_type}: {utilization:.1f}% ({len(eq_tasks)} tasks)")
    
    # Delay analysis
    delayed_tasks = df_solution[df_solution['delay'] > 0]
    on_time_tasks = df_solution[df_solution['delay'] == 0]
    
    print(f"\n⏰ DELAY ANALYSIS:")
    print(f"  On-time tasks: {len(on_time_tasks)} ({len(on_time_tasks)/len(df_solution)*100:.1f}%)")
    print(f"  Delayed tasks: {len(delayed_tasks)} ({len(delayed_tasks)/len(df_solution)*100:.1f}%)")
    
    if not delayed_tasks.empty:
        avg_delay = delayed_tasks['delay'].mean()
        max_delay = delayed_tasks['delay'].max()
        print(f"  Average delay: {avg_delay:.1f} minutes")
        print(f"  Maximum delay: {max_delay:.1f} minutes")
    
    # Timeline analysis
    print(f"\n📅 TIMELINE ANALYSIS:")
    makespan = df_solution['end_time'].max()
    total_processing_time = df_solution['processing_time'].sum()
    efficiency = (total_processing_time / makespan) * 100 if makespan > 0 else 0
    
    print(f"  Makespan: {makespan:.1f} minutes")
    print(f"  Total processing time: {total_processing_time:.1f} minutes")
    print(f"  Efficiency: {efficiency:.1f}%")
    
    return df_solution

# Analyze the best heuristic solution
best_solution_df = analyze_heuristic_solution(best_heuristic[1], best_heuristic[0])

### Visualization of Heuristic Results

In [ ]:
def visualize_heuristic_comparison(results):
    """Create comprehensive comparison visualizations"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Smart Container Terminal - Heuristic Comparison', fontsize=16, fontweight='bold')
    
    # Extract data for plotting
    names = [r[0] for r in results]
    costs = [r[2] for r in results]
    times = [r[3] for r in results]
    solutions = [r[1] for r in results]
    
    # 1. Cost Comparison
    ax1 = axes[0, 0]
    bars1 = ax1.bar(names, costs, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7)
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Solution Cost Comparison')
    ax1.grid(True, alpha=0.3)
    
    # Add cost values on bars
    for bar, cost in zip(bars1, costs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(costs)*0.01, 
                f'${cost:.0f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Solve Time Comparison
    ax2 = axes[0, 1]
    bars2 = ax2.bar(names, times, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7)
    ax2.set_ylabel('Solve Time (seconds)')
    ax2.set_title('Computational Time Comparison')
    ax2.grid(True, alpha=0.3)
    
    # Add time values on bars
    for bar, time_val in zip(bars2, times):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.01, 
                f'{time_val:.4f}s', ha='center', va='bottom', fontweight='bold')
    
    # 3. Equipment Utilization Comparison
    ax3 = axes[1, 0]
    
    utilization_data = []
    for name, solution in zip(names, solutions):
        if solution:
            df_sol = pd.DataFrame(solution)
            for eq_type in ['AGV', 'QC', 'YC']:
                eq_tasks = df_sol[df_sol['equipment_type'] == eq_type]
                if not eq_tasks.empty:
                    total_processing = eq_tasks['processing_time'].sum()
                    eq_count = len([e for e in equipment if e.type == eq_type])
                    utilization = (total_processing / (eq_count * 1440)) * 100
                    utilization_data.append({
                        'Heuristic': name,
                        'Equipment': eq_type,
                        'Utilization': utilization
                    })
    
    if utilization_data:
        util_df = pd.DataFrame(utilization_data)
        pivot_df = util_df.pivot(index='Equipment', columns='Heuristic', values='Utilization')
        
        pivot_df.plot(kind='bar', ax=ax3, alpha=0.7, width=0.8)
        ax3.set_ylabel('Utilization (%)')
        ax3.set_title('Equipment Utilization by Heuristic')
        ax3.legend(title='Heuristic')
        ax3.grid(True, alpha=0.3)
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
    
    # 4. Delay Analysis Comparison
    ax4 = axes[1, 1]
    
    delay_data = []
    for name, solution in zip(names, solutions):
        if solution:
            df_sol = pd.DataFrame(solution)
            delayed_tasks = df_sol[df_sol['delay'] > 0]
            delay_percentage = (len(delayed_tasks) / len(df_sol)) * 100
            avg_delay = delayed_tasks['delay'].mean() if not delayed_tasks.empty else 0
            
            delay_data.append({
                'Heuristic': name,
                'Delay_Percentage': delay_percentage,
                'Average_Delay': avg_delay
            })
    
    if delay_data:
        delay_df = pd.DataFrame(delay_data)
        
        # Create dual axis for delay percentage and average delay
        ax4_twin = ax4.twinx()
        
        bars4 = ax4.bar(delay_df['Heuristic'], delay_df['Delay_Percentage'], 
                       color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7, width=0.6)
        line4 = ax4_twin.plot(delay_df['Heuristic'], delay_df['Average_Delay'], 
                             'ro-', linewidth=2, markersize=8, label='Avg Delay (min)')
        
        ax4.set_ylabel('Delayed Tasks (%)', color='black')
        ax4_twin.set_ylabel('Average Delay (minutes)', color='red')
        ax4.set_title('Delay Analysis by Heuristic')
        ax4.grid(True, alpha=0.3)
        ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha='right')
        
        # Add percentage values on bars
        for bar, pct in zip(bars4, delay_df['Delay_Percentage']):
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(delay_df['Delay_Percentage'])*0.01, 
                    f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize heuristic comparison
visualize_heuristic_comparison(heuristic_results)

### Scalability Analysis

Let's test how the heuristics perform with larger problem instances:

In [ ]:
def scalability_test():
    """Test heuristic performance on larger instances"""
    
    print("📈 SCALABILITY ANALYSIS")
    print("=" * 40)
    
    # Test different problem sizes
    test_sizes = [
        (10, 3, 2, 3),  # 10 containers, 3 AGVs, 2 QCs, 3 YCs
        (20, 5, 3, 4),  # 20 containers, 5 AGVs, 3 QCs, 4 YCs
        (30, 7, 4, 5),  # 30 containers, 7 AGVs, 4 QCs, 5 YCs
    ]
    
    scalability_results = []
    
    for num_containers, num_agv, num_qc, num_yc in test_sizes:
        print(f"\n🔍 Testing {num_containers} containers, {num_agv} AGVs, {num_qc} QCs, {num_yc} YCs")
        print("-" * 60)
        
        # Generate larger instance
        # Simplified instance generation for scalability test
        test_containers = []
        for i in range(num_containers):
            test_containers.append(Container(
                id=i+1,
                size='20ft',
                weight=20,
                origin='yard' if i % 2 == 0 else 'vessel',
                destination='vessel' if i % 2 == 0 else 'yard',
                priority=1,
                earliest_time=0,
                latest_time=120
            ))
        
        test_equipment = []
        eq_id = 1
        
        # AGVs
        for i in range(num_agv):
            test_equipment.append(Equipment(
                id=eq_id, type='AGV', capacity=40, speed=200,
                position=(i*100, 150), available_times=list(range(1440))
            ))
            eq_id += 1
        
        # QCs
        for i in range(num_qc):
            test_equipment.append(Equipment(
                id=eq_id, type='QC', capacity=50, speed=2,
                position=(200 + i*100, 50), available_times=list(range(1440))
            ))
            eq_id += 1
        
        # YCs
        for i in range(num_yc):
            test_equipment.append(Equipment(
                id=eq_id, type='YC', capacity=40, speed=1.5,
                position=(100 + i*100, 200), available_times=list(range(1440))
            ))
            eq_id += 1
        
        # Generate tasks
        test_tasks = []
        task_id = 1
        
        for container in test_containers:
            # Simplified task generation
            pickup_type = 'YC' if container.origin == 'yard' else 'QC'
            delivery_type = 'QC' if container.destination == 'vessel' else 'YC'
            
            # 3 tasks per container
            test_tasks.append(Task(task_id, container.id, pickup_type, 5, (100, 200), []))
            task_id += 1
            
            test_tasks.append(Task(task_id, container.id, 'AGV', 10, (200, 150), [task_id-1]))
            task_id += 1
            
            test_tasks.append(Task(task_id, container.id, delivery_type, 5, (300, 200), [task_id-1]))
            task_id += 1
        
        # Test heuristics
        heuristic_times = []
        heuristic_costs = []
        
        # Test priority-based (fastest)
        _, cost1, time1 = priority_based_dispatching(test_containers, test_equipment, test_tasks)
        heuristic_times.append(time1)
        heuristic_costs.append(cost1)
        
        # Test greedy assignment
        _, cost2, time2 = greedy_assignment_balancing(test_containers, test_equipment, test_tasks)
        heuristic_times.append(time2)
        heuristic_costs.append(cost2)
        
        # Test look-ahead (might be slower)
        _, cost3, time3 = look_ahead_scheduling(test_containers, test_equipment, test_tasks, look_ahead_window=3)
        heuristic_times.append(time3)
        heuristic_costs.append(cost3)
        
        scalability_results.append({
            'Containers': num_containers,
            'Tasks': len(test_tasks),
            'Equipment': len(test_equipment),
            'Priority_Time': time1,
            'Greedy_Time': time2,
            'LookAhead_Time': time3,
            'Priority_Cost': cost1,
            'Greedy_Cost': cost2,
            'LookAhead_Cost': cost3
        })
        
        print(f"  Priority-Based: {time1:.4f}s, ${cost1:.0f}")
        print(f"  Greedy + Balancing: {time2:.4f}s, ${cost2:.0f}")
        print(f"  Look-Ahead: {time3:.4f}s, ${cost3:.0f}")
    
    # Create scalability visualization
    df_scalability = pd.DataFrame(scalability_results)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Solve Time vs Problem Size
    ax1.plot(df_scalability['Tasks'], df_scalability['Priority_Time'], 'bo-', label='Priority-Based', linewidth=2, markersize=8)
    ax1.plot(df_scalability['Tasks'], df_scalability['Greedy_Time'], 'go-', label='Greedy + Balancing', linewidth=2, markersize=8)
    ax1.plot(df_scalability['Tasks'], df_scalability['LookAhead_Time'], 'ro-', label='Look-Ahead', linewidth=2, markersize=8)
    
    ax1.set_xlabel('Number of Tasks')
    ax1.set_ylabel('Solve Time (seconds)')
    ax1.set_title('Scalability: Solve Time vs Problem Size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')  # Log scale for better visualization
    
    # Plot 2: Solution Quality vs Problem Size
    ax2.plot(df_scalability['Tasks'], df_scalability['Priority_Cost'], 'bo-', label='Priority-Based', linewidth=2, markersize=8)
    ax2.plot(df_scalability['Tasks'], df_scalability['Greedy_Cost'], 'go-', label='Greedy + Balancing', linewidth=2, markersize=8)
    ax2.plot(df_scalability['Tasks'], df_scalability['LookAhead_Cost'], 'ro-', label='Look-Ahead', linewidth=2, markersize=8)
    
    ax2.set_xlabel('Number of Tasks')
    ax2.set_ylabel('Solution Cost ($)')
    ax2.set_title('Scalability: Solution Quality vs Problem Size')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return df_scalability

# Run scalability analysis
scalability_results = scalability_test()

### Summary and Key Insights

#### Heuristic Implementation Achievements:
1. **Multiple Approaches**: Implemented three distinct heuristics with different strategies
2. **Fast Performance**: All heuristics solve problems in milliseconds vs seconds/minutes for MIP
3. **Scalability**: Heuristics handle larger instances effectively
4. **Quality Solutions**: Achieve reasonable solution quality with significant speed improvements

#### Heuristic Comparison:

**Priority-Based Dispatching:**
- **Strengths**: Fastest execution, easy to understand
- **Weaknesses**: May overlook load balancing and future impacts
- **Best for**: Real-time decisions where speed is critical

**Greedy Assignment with Load Balancing:**
- **Strengths**: Good balance between speed and quality, considers equipment workload
- **Weaknesses**: Still myopic, doesn't consider future task dependencies
- **Best for**: General-purpose scheduling with balanced performance

**Look-Ahead Scheduling:**
- **Strengths**: Considers future impacts, often better solution quality
- **Weaknesses**: Slower than other heuristics, more complex
- **Best for**: Strategic planning where solution quality is more important than speed

#### Key Insights:
1. **Speed vs Quality Trade-off**: Clear trade-off between computational speed and solution quality
2. **Problem Size Impact**: All heuristics scale well, but look-ahead becomes relatively slower on larger instances
3. **Equipment Coordination**: All heuristics successfully coordinate multiple equipment types
4. **Practical Applicability**: Heuristics provide practical solutions for real-time terminal operations

#### When to Use Each Heuristic:
- **Priority-Based**: Emergency situations, real-time dispatching
- **Greedy + Balancing**: Daily operations, balanced performance requirements
- **Look-Ahead**: Strategic planning, optimization-focused environments

These heuristics provide practical alternatives to the mathematical programming approach from Tier 1, offering different trade-offs between speed and solution quality for various operational scenarios.